In [ ]:

import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box
from scipy.spatial.distance import pdist,squareform
import seaborn as sns
import pytim
import warnings
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list
    
    
    
def surface_comp2(u, start=4000, stop=-1, step=10000):
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.5,
            max_layers=3,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        mid_resids = np.unique(layer2_atoms.resids)
        inner_resids = np.unique(layer3_atoms.resids)
        
        
        return inner_resids ,mid_resids,outer_resids,res_list
    

def surface_comp3(u, start=4000, stop=-1, step=10000):
    
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.2,
            max_layers=2,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        #layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        inner_resids = np.unique(layer2_atoms.resids)
        
        return inner_resids,outer_resids,res_list


def surface_comp_new(u):
    
    inner_layers_all = []
    outer_layers_all = []
    frame = u.trajectory[0]
    
    centroid, to_cluster, clustered_list, res_list = clustering(u,frame, eps=9)

    # --- Build selection for clustered core residues ---
    selection = 'resindex '
    for item in res_list:
        selection += str(item) + ' or resindex '
    selection = selection[0:-13]

    sel = u.select_atoms(selection)

    # --- Write clustered residues to temp PDB and reload ---
    with mda.Writer('spatial.pdb') as pdb:
        pdb.write(sel)
    univ = mda.Universe('spatial.pdb')

    # --- Run GITIM surface analysis ---
    g = univ.select_atoms('name CA')
    interface = pytim.GITIM(
        univ,
        group=g,
        alpha=3.9,
        max_layers=2,
        molecular=False,
        centered='middle'
    )

    # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
    layer1_atoms = interface.layers[0]   # outer surface
    layer2_atoms = interface.layers[1]   # mid layer
    #layer3_atoms = interface.layers[2]   # inner layer

    outer_resids = np.unique(layer1_atoms.resids)
    #inner_resids = np.unique(layer2_atoms.resids)
     
    inner_resids = np.setdiff1d(res_list, outer_resids)

    print(len(outer_resids), len(inner_resids), len(res_list))

    return inner_resids, outer_resids, res_list




In [ ]:
#!/usr/bin/env python
# coding: utf-8


def cos_sim(a, b):
    """Takes 2 vectors a, b and returns the cosine similarity 
    """
    dot_product = np.dot(a, b) # x.y
    norm_a = np.linalg.norm(a) #|x|
    norm_b = np.linalg.norm(b) #|y|
    return dot_product / (norm_a * norm_b)

def dist_array(arr):
    """takes an array of positions (another array of length 2)  and 
    return an arryay composed of the norm of the positions and the positions
    eg; arr[N][3] is the input
    output will be Array[N][4] = [d, x, y, z]
    """
    ret_arr = np.zeros((len(arr),4))
    for i in range(len(arr)):
        ret_arr[i][0]=np.linalg.norm(arr[i])
        ret_arr[i][1]=arr[i][0]
        ret_arr[i][2]=arr[i][1]
        ret_arr[i][3]=arr[i][2]
    
    return ret_arr

"""Calculate the translational tetrahedral order 
parameter given the array containing the distances of the 
four adjacent water oxygens to the the water (oxygen) of interest"""

def S_k(arr):
    return 1-(1.0/3.0)*(((np.var(d_4, dtype=np.float32))/2*np.mean(d_4, dtype=np.float32))**2)

def S_k(arr):
    arr = np.array(arr, dtype=np.float32)

    mean_d = np.mean(arr)
    var_d = np.var(arr)

    ratio = var_d / (2.0 * mean_d)

    return 1.0 - (1.0 / 3.0) * (ratio ** 2)

In [ ]:
systems = ['charged','equal','polar','nonpolar','aromatic']

#systems = ['2_close','2_spread','3_close','3_spread']

files = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']

#files = ['2_close','2_spread','3_close','3_spread']

layer_name = ['inner','outer','bulk']

for num,sys in enumerate(systems):
    
    stru = parmed.load_file(f'../gxg_comps/3_res/{sys}/output.top',
                       xyz=f'../gxg_comps/3_res/{sys}/output.gro')
    
    name = files[num]

    u = mda.Universe(stru,
                     f'frame2_{name}.pdb',
                     #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                    )
    
    inner_layers,outer_layers,res_list = surface_comp_new(u)

    all_list = ' '
    for item in res_list:
        all_list += str(item) + ' '



    inner_layer_list = ' '
    for item in inner_layers:
        inner_layer_list += str(item) + ' '


    outer_layer_list = ' '
    for item in outer_layers:
        outer_layer_list += str(item) + ' '


    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]


    dist_water_inner = 3.3
    dist_water_outer = 3.0

    #u.trajectory[100000]

    # Step 2: Water within dist_water of the CORE (clustered) residues

    select_inner = f" name OW  and (around {dist_water_inner} (protein and resid {inner_layer_list}))"

    select_outer = f" name OW  and (around {dist_water_outer} (protein and resid {outer_layer_list}))"

    water_inner = u.select_atoms(select_inner)

    water_outer = u.select_atoms(select_outer)
    water_inner  = water_inner-water_outer
    
    bulk_water = f" name OW  and not (around {8} (protein))"
    water_bulk = u.select_atoms(bulk_water)
    
    for name,lay in enumerate([water_inner,water_outer,water_bulk]):
        
        r_max = 3.9000
        with open(f'Tet_OP_3_{sys}_{layer_name[name]}.txt', 'w') as f:
            for hyd_water in lay:
                ne_water = u.select_atoms(f'name OW and around {r_max} index {hyd_water.index}')

                if(len(ne_water)) < 4: 
                    continue
                else:
                    R_n = ne_water.positions
                    R_0 = np.zeros(R_n.shape,dtype=np.float32)
                    for i in range(len(R_n)):
                        for j in range(3):
                            R_0[i][j] = hyd_water.position[j]
                    R_rel = R_n - R_0
                    dist_R_rel = dist_array(R_rel)
                    dist_nearest_4 = dist_R_rel[dist_R_rel[:,0].argsort()][0:4]
                    R_4 = dist_nearest_4[:,1:]
                    d_4 = dist_nearest_4[:,0:1]
                    """orientational tetrahedral order parameter"""
                    cos_sum = 0.0
                    for i in range(3):
                        for j in range(i+1,4,1):
                            cos = cos_sim(R_4[i],R_4[j])
                            cos_sum = cos_sum + (cos + 1.0/3.0)*(cos + 1.0/3.0)
                    q = 1.0 - (3.0/8.0)*cos_sum


                    f.write(f'\t{S_k(d_4)}\t{q}')
                    f.write('\n')    


In [ ]:

systems = ['3_charged','3_equal','3_polar','3_nonpolar','3_aromatic',
            'charged', 'equal', 'polar', 'nonpolar', 'aromatic',
           '2_close','2_spread','3_close','3_spread'
          ]


systems = ['charged','equal','polar','nonpolar','aromatic']


# ---- Load and collect means ----
for sys in systems[:]:

    data_inner = np.loadtxt(f'Tet_OP_{sys}_inner.txt')
    data_outer = np.loadtxt(f'Tet_OP_{sys}_outer.txt')   # fixed
    data_bulk  = np.loadtxt(f'Tet_OP_{sys}_bulk.txt')

    q_inner = data_inner[:, 1]
    q_outer = data_outer[:, 1]
    q_bulk  = data_bulk[:, 1]

    q_inner_all.append(np.mean(q_inner))
    q_outer_all.append(np.mean(q_outer))
    q_bulk_all.append(np.mean(q_bulk))

# ---- Bar plot ----
x = np.arange(len(systems))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(x - width, q_inner_all, width, label='Inner')
ax.bar(x,         q_outer_all, width, label='Outer')
ax.bar(x + width, q_bulk_all,  width, label='Bulk')

ax.set_xticks(x)
ax.set_xticklabels(systems, rotation=20)

ax.set_ylabel('Average Tetrahedral Order Parameter')
ax.set_title('Comparison of Water Order Parameters')
ax.legend()
ax.set_ylim(0.28,0.54)
plt.tight_layout()
plt.show()


## Dihedral Constant 

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis.dielectric import DielectricConstant

In [ ]:
files = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']
files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']
systems = ['charged',
           'equal','polar','nonpolar','aromatic'
          ]
results = []
#files = ['2_close','2_spread','3_close','3_spread']

layer_name = ['inner','outer','bulk']

for num,sys in enumerate(systems):
    
    stru = parmed.load_file(f'../gxg_comps/9_res/{sys}/output.top',
                       xyz=f'../gxg_comps/9_res/{sys}/output.gro')
    
    name = files[num]

    u = mda.Universe(stru,
                     #f'frame2_{name}.pdb',
                     f'../gxg_comps/9_res/{sys}/traj.dcd'
                    )
    
    traj_start=115000
    
    inner_layers, outer_layers, res_list = surface_comp3(
        u,
        #type_u=3,
        start=traj_start, stop=traj_start+1, step=1
    )
    

    all_list = ' '
    for item in res_list:
        all_list += str(item) + ' '



    inner_layer_list = ' '
    for item in inner_layers:
        inner_layer_list += str(item) + ' '


    outer_layer_list = ' '
    for item in outer_layers:
        outer_layer_list += str(item) + ' '


    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]


    dist_water_inner = 3.3
    dist_water_outer = 3.0

    select_inner = f"same residue as (resname SOL and (around {dist_water_inner} (protein and resid {inner_layer_list})))"
    water_inner_raw = u.select_atoms(select_inner)

    select_outer = f"same residue as (resname SOL and (around {dist_water_outer} (protein and resid {outer_layer_list})))"
    water_outer_raw = u.select_atoms(select_outer)

    bulk_water = f"same residue as (resname SOL and not (around 8 (protein)))"
    water_bulk = u.select_atoms(bulk_water)
    
    inner_only_resids = set(water_inner_raw.resids) - set(water_outer_raw.resids)
    resid_str = ' '.join(map(str, inner_only_resids))
    water_inner = u.select_atoms(f"same residue as (resname SOL and resid {resid_str})")

    diel_inner = DielectricConstant(water_inner, temperature=300)
    diel_inner.run(start=traj_start, stop=traj_start+70, step=1)
    
    diel_outer = DielectricConstant(water_outer_raw, temperature=300)
    diel_outer.run(start=traj_start, stop=traj_start+70, step=1)
    
    diel_bulk = DielectricConstant(water_bulk, temperature=300)
    diel_bulk.run(start=traj_start, stop=traj_start+70, step=1)
    
    results.append({
        "system": files[num],
        "inner": diel_inner.results.eps_mean,
        "outer": diel_outer.results.eps_mean,
        "bulk": diel_bulk.results.eps_mean
    })

# Save
df = pd.DataFrame(results)
df.to_csv("dielectric_constants.csv", index=False)

print(df)



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("dielectric_constants.csv")

x = np.arange(len(df))
width = 0.25

fig, ax = plt.subplots(figsize=(8,5))

ax.bar(x - width, df["inner"], width, label="Inner")
ax.bar(x,         df["outer"], width, label="Outer")
ax.bar(x + width, df["bulk"],  width, label="Bulk")

ax.set_xticks(x)
ax.set_xticklabels(df["system"], rotation=45)
ax.set_ylabel("Dielectric Constant")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
## Whole Systems 


eps_clust  = 9
#files = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']
files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']
#files = ['2_close','2_spread','3_close','3_spread']
systems = ['charged',
           'equal','polar',
           'nonpolar','aromatic'
          ]

results = []
#files = ['2_close','2_spread','3_close','3_spread']

frames = [78000,77000,79000,80000,81000,
          70000,85000,90000,92000,95000,
          100000,105000,
          110000,112000]

for num, sys in enumerate(systems):
#     stru = parmed.load_file(
#         f'../gxg_comps/3_res/{sys}/output.top',
#         xyz=f'../gxg_comps/3_res/{sys}/output.gro'
#     )

#     u = mda.Universe(
#         stru,
#         f'../gxg_comps/3_res/{sys}/trajectory.dcd'
#     )
    
    stru = parmed.load_file(
        f'../gxg_comps/9_res/{sys}/output.top',
        xyz=f'../gxg_comps/9_res/{sys}/output.gro'
    )

    u = mda.Universe(
        stru,
        f'../gxg_comps/9_res/{sys}/traj.dcd'
    )

    eps_values = []

    for no_frame in frames:

        traj_start = no_frame
        ts = u.trajectory[traj_start]

        centroid, to_cluster, clustered_list, res_list = clustering(
            u, ts, eps=eps_clust
        )

        all_list = " ".join(map(str, res_list))

        dist_water = 3.3

        select = (
            f"same residue as "
            f"(resname SOL and "
            f"(around {dist_water} (protein and resid {all_list})))"
        )

        water_all = u.select_atoms(select)

        diel_ = DielectricConstant(water_all, temperature=300)
        diel_.run(start=traj_start, stop=traj_start + 50, step=1)

        eps_values.append(diel_.results.eps_mean)

    results.append({
        "system": files[num],
        "mean_eps": np.mean(eps_values),
        "std_eps": np.std(eps_values),
        "n_frames": len(eps_values)
    })

df = pd.DataFrame(results)
df.to_csv("dielectric_constants_all_9res.csv", index=False)
print(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("dielectric_constants_all_9res.csv")

x = np.arange(len(df))

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    x,
    df["mean_eps"],
    yerr=df["std_eps"],
    capsize=5,
)

ax.set_xticks(x)
ax.set_xticklabels(df["system"], rotation=45)
ax.set_ylabel("Dielectric Constant")
ax.set_xlabel("System")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df_3 = pd.read_csv("dielectric_constants_all_3res.csv")
df_9 = pd.read_csv("dielectric_constants_all_9res.csv")
df = pd.concat([df_3, df_9], ignore_index=True)

x = np.arange(len(df))

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    x,
    df["mean_eps"],
    yerr=df["std_eps"],
    capsize=5,
)

ax.set_xticks(x)
ax.set_xticklabels(df["system"], rotation=45)
ax.set_ylabel("Dielectric Constant")
ax.set_xlabel("System")
ax.set_ylim(2.0,3.0)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("dielectric_constants_all_9res.csv")

x = np.arange(len(df))

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    x,
    df["mean_eps"],
    yerr=df["std_eps"],
    capsize=5,
)

ax.set_xticks(x)
ax.set_xticklabels(df["system"], rotation=45)
ax.set_ylabel("Dielectric Constant")
ax.set_xlabel("System")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("dielectric_constants_all_3res.csv")

x = np.arange(len(df))
width = 0.25

fig, ax = plt.subplots(figsize=(8,5))

ax.bar(x, df["all"], width, label="Inner")


ax.set_xticks(x)
ax.set_xticklabels(df["system"], rotation=45)
ax.set_ylabel("Dielectric Constant")
ax.legend()

plt.tight_layout()
plt.show()

### rdf from coms of condensate

In [ ]:
systems = ['charged','equal','polar','nonpolar','aromatic']

#systems = ['2_close','2_spread','3_close','3_spread']

files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']

#files = ['2_close','2_spread','3_close','3_spread']

#layer_name = ['inner','outer','bulk']

for num,sys in enumerate(systems):
    
    stru = parmed.load_file(f'../gxg_comps/9_res/{sys}/output.top',
                       xyz=f'../gxg_comps/9_res/{sys}/output.gro')
    
    name = files[num]

    u = mda.Universe(stru,
                     f'frame2_{name}.pdb',
                     #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                    )
    
    
    rdf_all = []

    # RDF settings
    r_max = 50.0
    dr = 5
    bins = np.arange(0, r_max + dr, dr)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    
    centroid, to_cluster, clustered_list, res_list = clustering(
    u, ts, eps=9)
    

    # centroid of condensate
    center = np.array(centroid)

    # =====================================================
    # SELECT ALL HEAVY ATOMS
    # =====================================================
#     heavy_atoms = u.select_atoms(
#         "(protein or nucleic) and not name H*"
#     )
    heavy_atoms = u.select_atoms(
        "(name OW HW1 HW2)"
    )    

    positions = heavy_atoms.positions

    # =====================================================
    # DISTANCE FROM CONDENSATE CENTER
    # =====================================================
    dist = np.linalg.norm(positions - center, axis=1)

    # =====================================================
    # HISTOGRAM
    # =====================================================
    hist, edges = np.histogram(dist, bins=bins)

    # spherical shell volume
    shell_vol = (4.0 / 3.0) * np.pi * (
        edges[1:]**3 - edges[:-1]**3
    )

    # number density
    box_vol = np.prod(u.dimensions[:3])
    density = len(heavy_atoms) / box_vol

    # RDF
    g_r = hist / (shell_vol * density)

    # save frame rdf
    temp_df = pd.DataFrame({
        "frame": ts.frame,
        "r": bin_centers,
        "g_r": g_r
    })

    rdf_all.append(temp_df)

    # =====================================================
    # SAVE CSV
    # =====================================================
    rdf_df = pd.concat(rdf_all, ignore_index=True)

    rdf_df.to_csv(f"condensate_center_rdf_water_{name}.csv", index=False)

    print(f"Saved RDF to {name}")    
    
    

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

systems = ['charged','equal','polar','nonpolar','aromatic']
files = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']
files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']

# =====================================================
# FIGURE
# =====================================================
fig, ax = plt.subplots(figsize=(5,4))

# =====================================================
# LOOP THROUGH SYSTEMS
# =====================================================
for num, sys in enumerate(systems):

    name = files[num]

    # Load RDF data
    df_ow = pd.read_csv(f"condensate_center_rdf_protein_{name}.csv")

    # Average g(r)
    avg_rdf_ow = (
        df_ow.groupby("r", as_index=False)["g_r"]
        .mean()
    )

    # Plot
    ax.plot(
        avg_rdf_ow["r"],
        avg_rdf_ow["g_r"],
        lw=2,
        label=sys
    )

# =====================================================
# LABELS
# =====================================================
ax.set_xlabel(r"r ($\AA$)", fontsize=12)
ax.set_ylabel(r"g(r)", fontsize=12)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()
#systems = ['2_close','2_spread','3_close','3_spread']

files = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']

#files = ['2_close','2_spread','3_close','3_spread']

#layer_name = ['inner','outer','bulk']

for num,sys in enumerate(systems):
    name = files[num]
    rdf_df= pd.read_csv(f"condensate_center_rdf_water_{name}.csv")
    
    

### RDF Ow-OW 

In [ ]:
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis.rdf import InterRDF
import parmed

# ============================================================
# SYSTEMS
# ============================================================

systems = ['charged','equal','polar','nonpolar','aromatic']
files   = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']
files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']
# ============================================================
# RDF SETTINGS
# ============================================================

rdf_range = (0.0, 20.0)
nbins = 200

dist_water_inner = 3.3
dist_water_outer = 3.0

# ============================================================
# LOOP OVER SYSTEMS
# ============================================================

for num, sys in enumerate(systems):

    print(f"\nProcessing system: {sys}")

    # ========================================================
    # LOAD TOPOLOGY + TRAJECTORY
    # ========================================================

    stru = parmed.load_file(
        f'../gxg_comps/9_res/{sys}/output.top',
        xyz=f'../gxg_comps/9_res/{sys}/output.gro'
    )

    name = files[num]

    u = mda.Universe(
        stru,
        f'frame2_{name}.pdb'
        # '../gxg_comps/3_res/charged/traj_2-nw.dcd'
    )

    # ========================================================
    # GET SURFACE LAYERS
    # ========================================================

    inner_layers, outer_layers, res_list = surface_comp_new(u)

    # resid strings
    inner_layer_list = ' '.join(map(str, inner_layers))
    outer_layer_list = ' '.join(map(str, outer_layers))

    # ========================================================
    # WATER SELECTIONS
    # ========================================================

    # Inner shell waters
    select_inner = (
        f"name OW and "
        f"(around {dist_water_inner} "
        f"(protein and resid {inner_layer_list}))"
    )

    # Outer shell waters
    select_outer = (
        f"name OW and "
        f"(around {dist_water_outer} "
        f"(protein and resid {outer_layer_list}))"
    )

    water_inner = u.select_atoms(select_inner)
    water_outer = u.select_atoms(select_outer)

    # Remove overlap
    water_inner = water_inner - water_outer

    # Bulk waters
    bulk_water = "name OW and not (around 8 protein)"
    water_bulk = u.select_atoms(bulk_water)

    # All waters
    water_all = u.select_atoms("name OW")
    water_all = u.select_atoms("protein")

    # ========================================================
    # PRINT COUNTS
    # ========================================================

    print("Inner waters :", len(water_inner))
    print("Outer waters :", len(water_outer))
    print("Bulk waters  :", len(water_bulk))
    print("All waters   :", len(water_all))

    # ========================================================
    # RDF CALCULATIONS
    # ========================================================

    print("Calculating RDFs...")

    rdf_inner = InterRDF(
        water_inner,
        water_all,
        nbins=nbins,
        range=rdf_range
    )

    rdf_outer = InterRDF(
        water_outer,
        water_all,
        nbins=nbins,
        range=rdf_range
    )

    rdf_bulk = InterRDF(
        water_bulk,
        water_all,
        nbins=nbins,
        range=rdf_range
    )

    rdf_inner.run()
    rdf_outer.run()
    rdf_bulk.run()

    # ========================================================
    # SAVE RDF DATA
    # ========================================================

    df_inner = pd.DataFrame({
        'r': rdf_inner.results.bins,
        'g_r': rdf_inner.results.rdf
    })

    df_outer = pd.DataFrame({
        'r': rdf_outer.results.bins,
        'g_r': rdf_outer.results.rdf
    })

    df_bulk = pd.DataFrame({
        'r': rdf_bulk.results.bins,
        'g_r': rdf_bulk.results.rdf
    })

    # ========================================================
    # WRITE CSV FILES
    # ========================================================

    df_inner.to_csv(
        f'rdf_inner_Protein_{name}.csv',
        index=False
    )

    df_outer.to_csv(
        f'rdf_outer_Protein_{name}.csv',
        index=False
    )

    df_bulk.to_csv(
        f'rdf_bulk_Protein_{name}.csv',
        index=False
    )

    print(f"Saved RDF csv files for {sys}")

print("\nAll RDF calculations completed.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SYSTEMS
# ============================================================

systems = ['charged', 'equal', 'polar', 'nonpolar', 'aromatic']
files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']
# ============================================================
# COLOR MAP FOR WATER TYPES
# ============================================================

water_colors = {
    'inner': '#D62728',
    'outer': '#1F77B4',
    'bulk' : '#2CA02C',
}

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1, 5,
    figsize=(16, 3.5),
    sharex=True,
    sharey=True
)

# ============================================================
# LOAD + PLOT
# ============================================================

for ax, sys in zip(axes,files):

    # --------------------------------------------------------
    # LOAD CSV FILES
    # --------------------------------------------------------

    df_inner = pd.read_csv(f'rdf_inner_Protein_{sys}.csv')
    df_outer = pd.read_csv(f'rdf_outer_Protein_{sys}.csv')
    df_bulk  = pd.read_csv(f'rdf_bulk_Protein_{sys}.csv')

    # --------------------------------------------------------
    # PLOT INNER
    # --------------------------------------------------------

    ax.plot(
        df_inner['r'],
        df_inner['g_r'],
        lw=2,
        color=water_colors['inner'],
        label='Inner'
    )

    # --------------------------------------------------------
    # PLOT OUTER
    # --------------------------------------------------------

    ax.plot(
        df_outer['r'],
        df_outer['g_r'],
        lw=2,
        color=water_colors['outer'],
        label='Outer'
    )

    # --------------------------------------------------------
    # PLOT BULK
    # --------------------------------------------------------

    ax.plot(
        df_bulk['r'],
        df_bulk['g_r'],
        lw=2,
        color=water_colors['bulk'],
        label='Bulk'
    )

    # --------------------------------------------------------
    # FORMATTING
    # --------------------------------------------------------

    ax.set_title(sys.capitalize(), fontsize=12)

#     ax.set_xlim(2, 15)
#     ax.set_ylim(0, 4)

    ax.set_xlabel(r'r ($\AA$)', fontsize=10)

    ax.tick_params(
        axis='both',
        labelsize=9,
        direction='in'
    )

    ax.axhline(
        1.0,
        ls='--',
        lw=1,
        color='gray',
        alpha=0.6
    )

# ============================================================
# Y LABEL
# ============================================================

axes[0].set_ylabel(r'g(r)', fontsize=11)

# ============================================================
# LEGEND
# ============================================================

axes[-1].legend(
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

# ============================================================
# LAYOUT
# ============================================================

plt.tight_layout()

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    'rdf_systemwise_comparison.png',
    dpi=300,
    bbox_inches='tight'
)
# plt.ylim(0,3)
# plt.xlim(2,10)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SYSTEMS
# ============================================================

systems = ['charged', 'equal', 'polar', 'nonpolar', 'aromatic']
files = ['9_charge','9_equal','9_polar','9_nonpolar','9_aromatic']

# ============================================================
# COLOR MAP
# ============================================================

color_map = {
    '9_charge'  : '#D62728',
    '9_equal'    : '#1F77B4',
    '9_polar'    : '#2CA02C',
    '9_nonpolar' : '#9467BD',
    '9_aromatic' : '#FF7F0E',
}

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(12, 3.5),
    sharex=True,
    sharey=True
)

ax_inner, ax_outer, ax_bulk = axes

# ============================================================
# LOAD + PLOT
# ============================================================

for sys in files:

    # --------------------------------------------------------
    # LOAD CSV FILES
    # --------------------------------------------------------

    df_inner = pd.read_csv(f'rdf_inner_Protein_{sys}.csv')
    df_outer = pd.read_csv(f'rdf_outer_Protein_{sys}.csv')
    df_bulk  = pd.read_csv(f'rdf_bulk_Protein_{sys}.csv')

    # --------------------------------------------------------
    # INNER RDF
    # --------------------------------------------------------

    ax_inner.plot(
        df_inner['r'],
        df_inner['g_r'],
        lw=2,ls='--',
        color=color_map[sys],
        label=sys
    )

    # --------------------------------------------------------
    # OUTER RDF
    # --------------------------------------------------------

    ax_outer.plot(
        df_outer['r'],
        df_outer['g_r'],
        lw=1.4,ls='--',
        color=color_map[sys],
        label=sys
    )

    # --------------------------------------------------------
    # BULK RDF
    # --------------------------------------------------------

    ax_bulk.plot(
        df_bulk['r'],
        df_bulk['g_r'],
        lw=1.4,ls='--',
        color=color_map[sys],
        label=sys
    )

# ============================================================
# AXIS FORMATTING
# ============================================================

titles = ['Inner Water', 'Outer Water', 'Bulk Water']

for ax, title in zip(axes, titles):

    ax.set_title(title, fontsize=12)

    ax.set_xlabel(r'r ($\AA$)', fontsize=11)

    #ax.set_xlim(0, 15)

    ax.tick_params(
        axis='both',
        labelsize=10,
        direction='in'
    )

    ax.axhline(
        1.0,
        ls='--',
        lw=1,
        color='gray',
        alpha=0.6
    )

ax_inner.set_ylabel(r'g(r)', fontsize=11)

# ============================================================
# LEGEND
# ============================================================

ax_bulk.legend(
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

# ============================================================
# LAYOUT
# ============================================================

plt.tight_layout()

# ============================================================
# SAVE
# ============================================================
# plt.ylim(0,3)
# plt.xlim(2,10)
plt.savefig(
    'rdf_comparison_all_systems.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:

systems = ['charged','equal','polar','nonpolar','aromatic']

#systems = ['2_close','2_spread','3_close','3_spread']

files = ['3_charge','3_equal','3_polar','3_nonpolar','3_aromatic']

#files = ['2_close','2_spread','3_close','3_spread']

layer_name = ['inner','outer','bulk']

for num,sys in enumerate(systems):
    
    stru = parmed.load_file(f'../gxg_comps/3_res/{sys}/output.top',
                       xyz=f'../gxg_comps/3_res/{sys}/output.gro')
    
    name = files[num]

    u = mda.Universe(stru,
                     f'frame2_{name}.pdb',
                     #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                    )
    
    inner_layers,outer_layers,res_list = surface_comp_new(u)

    all_list = ' '
    for item in res_list:
        all_list += str(item) + ' '



    inner_layer_list = ' '
    for item in inner_layers:
        inner_layer_list += str(item) + ' '


    outer_layer_list = ' '
    for item in outer_layers:
        outer_layer_list += str(item) + ' '


    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]


    dist_water_inner = 3.3
    dist_water_outer = 3.0

    #u.trajectory[100000]

    # Step 2: Water within dist_water of the CORE (clustered) residues

    select_inner = f" name OW  and (around {dist_water_inner} (protein and resid {inner_layer_list}))"

    select_outer = f" name OW  and (around {dist_water_outer} (protein and resid {outer_layer_list}))"

    water_inner = u.select_atoms(select_inner)

    water_outer = u.select_atoms(select_outer)
    water_inner  = water_inner-water_outer
    
    bulk_water = f" name OW  and not (around {8} (protein))"
    water_bulk = u.select_atoms(bulk_water)
    
    
    water_all = f" name OW"
    
    
    ## Calculate rdf of water_inner vs water_all 
    ## Calculate rdf of water_outer vs water_all     
    ## Calculate rdf of water_bulk vs water_all     
    
    
    for name,lay in enumerate([water_inner,water_outer,water_bulk]):
        
        r_max = 3.9000
        with open(f'Tet_OP_3_{sys}_{layer_name[name]}.txt', 'w') as f:
            for hyd_water in lay:
                ne_water = u.select_atoms(f'name OW and around {r_max} index {hyd_water.index}')

                if(len(ne_water)) < 4: 
                    continue
                else:
                    R_n = ne_water.positions
                    R_0 = np.zeros(R_n.shape,dtype=np.float32)
                    for i in range(len(R_n)):
                        for j in range(3):
                            R_0[i][j] = hyd_water.position[j]
                    R_rel = R_n - R_0
                    dist_R_rel = dist_array(R_rel)
                    dist_nearest_4 = dist_R_rel[dist_R_rel[:,0].argsort()][0:4]
                    R_4 = dist_nearest_4[:,1:]
                    d_4 = dist_nearest_4[:,0:1]
                    """orientational tetrahedral order parameter"""
                    cos_sum = 0.0
                    for i in range(3):
                        for j in range(i+1,4,1):
                            cos = cos_sim(R_4[i],R_4[j])
                            cos_sum = cos_sum + (cos + 1.0/3.0)*(cos + 1.0/3.0)
                    q = 1.0 - (3.0/8.0)*cos_sum


                    f.write(f'\t{S_k(d_4)}\t{q}')
                    f.write('\n')    

                    

In [ ]:

import numpy as np
import pandas as pd
from MDAnalysis.analysis.distances import distance_array


stru = parmed.load_file('../gxg_comps/9_res/charged//output.top',
                       xyz='../gxg_comps/9_res/charged//output.gro')

name = '9_charge'
u = mda.Universe(stru,
                 f'frame2_{name}.pdb',
                 #'../gxg_comps/9_res/charged/traj.dcd'
                )

In [ ]:


rdf_all = []

# RDF settings
r_max = 45.0
dr = 3
bins = np.arange(0, r_max + dr, dr)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

for ts in u.trajectory[-1000::25]:

    # =====================================================
    # CLUSTERING
    # =====================================================
    centroid, to_cluster, clustered_list, res_list = clustering(
        u, ts, eps=9
    )

    # centroid of condensate
    center = np.array(centroid)

    # =====================================================
    # SELECT ALL HEAVY ATOMS
    # =====================================================
#     heavy_atoms = u.select_atoms(
#         "(protein or nucleic) and not name H*"
#     )
    heavy_atoms = u.select_atoms(
        "(name OW HW1 HW2)"
    )    

    positions = heavy_atoms.positions

    # =====================================================
    # DISTANCE FROM CONDENSATE CENTER
    # =====================================================
    dist = np.linalg.norm(positions - center, axis=1)

    # =====================================================
    # HISTOGRAM
    # =====================================================
    hist, edges = np.histogram(dist, bins=bins)

    # spherical shell volume
    shell_vol = (4.0 / 3.0) * np.pi * (
        edges[1:]**3 - edges[:-1]**3
    )

    # number density
    box_vol = np.prod(u.dimensions[:3])
    density = len(heavy_atoms) / box_vol

    # RDF
    g_r = hist / (shell_vol * density)

    # save frame rdf
    temp_df = pd.DataFrame({
        "frame": ts.frame,
        "r": bin_centers,
        "g_r": g_r
    })

    rdf_all.append(temp_df)

# =====================================================
# SAVE CSV
# =====================================================
rdf_df = pd.concat(rdf_all, ignore_index=True)

rdf_df.to_csv("condensate_center_rdf_water.csv", index=False)

print("Saved RDF to condensate_center_rdf.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_csv("condensate_center_rdf_protein_9_charge.csv")
df_ow = pd.read_csv("condensate_center_rdf_water_9_charge.csv")

# =====================================================
# AVERAGE RDF
# =====================================================

avg_rdf = (
    df.groupby("r", as_index=False)["g_r"]
    .mean()
)

avg_rdf_ow = (
    df_ow.groupby("r", as_index=False)["g_r"]
    .mean()
)

# =====================================================
# PLOT WITH TWO Y-AXES
# =====================================================

fig, ax1 = plt.subplots(figsize=(6,4))

# left y-axis
ax1.plot(avg_rdf["r"], avg_rdf["g_r"], label='Heavy atoms')
ax1.set_xlabel("r (Å)")
ax1.set_ylabel("Heavy atom g(r)")

# right y-axis
ax2 = ax1.twinx()

ax2.plot(avg_rdf_ow["r"], avg_rdf_ow["g_r"], label='Water RDF')
ax2.set_ylabel("Water g(r)")

plt.title("Average RDF around condensate center")

plt.show()

In [ ]:
for ts in u.trajectory:

    centroid, to_cluster, clustered_list, res_list = clustering(
        u, ts, eps=9
    )
         
    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]

    res_list = np.array(res_list)

    selection_2 = ' '
    for item in res_list:
            selection_2 += str(item)+ ' '       

    sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

    selection_3 = ' '
    for item in sel:
            selection_3 += str(item)+ ' '
            
## Calculate rdf of all heavy atoms around coms selection atoms 


In [ ]:

stru = parmed.load_file('../gxg_comps/9_res/nonpolar//output.top',
                       xyz='../gxg_comps/9_res/nonpolar//output.gro')

name = '9_nonpolar'
u = mda.Universe(stru,
                 f'frame2_{name}.pdb',
                 #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                )


In [ ]:

inner_layers,outer_layers,res_list = surface_comp_new(u)


all_list = ' '
for item in res_list:
    all_list += str(item) + ' '



inner_layer_list = ' '
for item in inner_layers:
    inner_layer_list += str(item) + ' '
    
    
outer_layer_list = ' '
for item in outer_layers:
    outer_layer_list += str(item) + ' '

    
selection = 'resindex '
for item in res_list:
        selection += str(item)+ ' or resindex '
selection = selection[0:-13]
    

dist_water_inner = 3.3
dist_water_outer = 3.0

#u.trajectory[100000]

# Step 2: Water within dist_water of the CORE (clustered) residues

select_inner = f" name OW  and (around {dist_water_inner} (protein and resid {inner_layer_list}))"

select_outer = f" name OW  and (around {dist_water_outer} (protein and resid {outer_layer_list}))"

water_inner = u.select_atoms(select_inner)

water_outer = u.select_atoms(select_outer)
water_inner  = water_inner-water_outer

In [ ]:
bulk_water = f" name OW  and not (around {8} (protein))"
bulk_water = u.select_atoms(bulk_water)


In [ ]:
water_inner.indices,water_outer.indices

In [ ]:
### Inner_water 

"""If the number of waters within 4.2A is less than 4 (coordination number),
skip the particular water molecule/step"""

r_max = 3.8000
with open('Tet_OP_inner.txt', 'w') as f:
    for hyd_water in water_inner:
        ne_water = u.select_atoms(f'name OW and around {r_max} index {hyd_water.index}')
        
        if(len(ne_water)) < 4: 
            continue
        else:
            R_n = ne_water.positions
            R_0 = np.zeros(R_n.shape,dtype=np.float32)
            for i in range(len(R_n)):
                for j in range(3):
                    R_0[i][j] = hyd_water.position[j]
            R_rel = R_n - R_0
            dist_R_rel = dist_array(R_rel)
            dist_nearest_4 = dist_R_rel[dist_R_rel[:,0].argsort()][0:4]
            R_4 = dist_nearest_4[:,1:]
            d_4 = dist_nearest_4[:,0:1]
            """orientational tetrahedral order parameter"""
            cos_sum = 0.0
            for i in range(3):
                for j in range(i+1,4,1):
                    cos = cos_sim(R_4[i],R_4[j])
                    cos_sum = cos_sum + (cos + 1.0/3.0)*(cos + 1.0/3.0)
            q = 1.0 - (3.0/8.0)*cos_sum


            f.write(f'\t{S_k(d_4)}\t{q}')
            f.write('\n')

In [ ]:
### outer_water 

"""If the number of waters within 4.2A is less than 4 (coordination number),
skip the particular water molecule/step"""

r_max = 3.8000
with open('Tet_OP_outer.txt', 'w') as f:
    for hyd_water in water_outer:
        ne_water = u.select_atoms(f'name OW and around {r_max} index {hyd_water.index}')
        
        if(len(ne_water)) < 4: 
            continue
        else:
            R_n = ne_water.positions
            R_0 = np.zeros(R_n.shape,dtype=np.float32)
            for i in range(len(R_n)):
                for j in range(3):
                    R_0[i][j] = hyd_water.position[j]
            R_rel = R_n - R_0
            dist_R_rel = dist_array(R_rel)
            dist_nearest_4 = dist_R_rel[dist_R_rel[:,0].argsort()][0:4]
            R_4 = dist_nearest_4[:,1:]
            d_4 = dist_nearest_4[:,0:1]
            """orientational tetrahedral order parameter"""
            cos_sum = 0.0
            for i in range(3):
                for j in range(i+1,4,1):
                    cos = cos_sim(R_4[i],R_4[j])
                    cos_sum = cos_sum + (cos + 1.0/3.0)*(cos + 1.0/3.0)
            q = 1.0 - (3.0/8.0)*cos_sum


            f.write(f'\t{S_k(d_4)}\t{q}')
            f.write('\n')

In [ ]:
### bulk_water 

"""If the number of waters within 4.2A is less than 4 (coordination number),
skip the particular water molecule/step"""

r_max = 3.8000
with open('Tet_OP_bulk.txt', 'w') as f:
    for hyd_water in bulk_water[::20]:
        ne_water = u.select_atoms(f'name OW and around {r_max} index {hyd_water.index}')
        
        if(len(ne_water)) < 4: 
            continue
        else:
            R_n = ne_water.positions
            R_0 = np.zeros(R_n.shape,dtype=np.float32)
            for i in range(len(R_n)):
                for j in range(3):
                    R_0[i][j] = hyd_water.position[j]
            R_rel = R_n - R_0
            dist_R_rel = dist_array(R_rel)
            dist_nearest_4 = dist_R_rel[dist_R_rel[:,0].argsort()][0:4]
            R_4 = dist_nearest_4[:,1:]
            d_4 = dist_nearest_4[:,0:1]
            """orientational tetrahedral order parameter"""
            cos_sum = 0.0
            for i in range(3):
                for j in range(i+1,4,1):
                    cos = cos_sim(R_4[i],R_4[j])
                    cos_sum = cos_sum + (cos + 1.0/3.0)*(cos + 1.0/3.0)
            q = 1.0 - (3.0/8.0)*cos_sum


            f.write(f'\t{S_k(d_4)}\t{q}')
            f.write('\n')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load data
data_inner = np.loadtxt('Tet_OP_inner.txt')
data_outer = np.loadtxt('Tet_OP_bulk.txt')

# Inner water
S_k_inner = data_inner[:, 0]
q_inner = data_inner[:, 1]

# Outer water
S_k_outer = data_outer[:, 0]
q_outer = data_outer[:, 1]

# Plot
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    S_k_inner,
    q_inner,
    s=10,
    alpha=0.5,
    label='Inner Water'
)

ax.scatter(
    S_k_outer,
    q_outer,
    s=10,
    alpha=0.5,
    label='Outer Water'
)

ax.set_xlabel(r'$S_k(d_4)$', fontsize=14)
ax.set_ylabel(r'Tetrahedral Order Parameter $q$', fontsize=14)

ax.set_title('Comparison of Inner and Outer Water')

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.hist(q_inner, density=True)
plt.hist(q_outer ,density=True)

In [ ]:
plt.hist(S_k_inner, density=True)
plt.hist(S_k_outer ,density=True)

In [ ]:
S_k_inner.mean(),S_k_outer.mean()

In [ ]:
q_inner.mean(),q_outer.mean()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load data
# Column 1 -> S_k(d_4)
# Column 2 -> tetrahedral order parameter q
data = np.loadtxt('Tet_OP_inner.txt')
data_outer = np.loadtxt('Tet_OP_outer.txt')
S_k = data[:, 0]
q = data[:, 1]

# Plot
fig, ax = plt.subplots(figsize=(6, 4))

ax.scatter(S_k, q, s=10, alpha=0.5)

ax.set_xlabel(r'$S_k(d_4)$', fontsize=14)
ax.set_ylabel(r'Tetrahedral Order Parameter $q$', fontsize=14)
ax.set_title('Tetrahedral Order Parameter vs $S_k(d_4)$')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('Tet_OP_inner.txt')

S_k = data[:, 0]
q = data[:, 1]

fig, ax = plt.subplots(figsize=(6, 4))

hist = ax.hist2d(S_k, q, bins=50)

plt.colorbar(hist[3], ax=ax, label='Counts')

ax.set_xlabel(r'$S_k(d_4)$', fontsize=14)
ax.set_ylabel(r'Tetrahedral Order Parameter $q$', fontsize=14)
ax.set_title('2D Distribution of Tetrahedral Order Parameter')

plt.tight_layout()
plt.show()

In [ ]:
water_inner

In [ ]:

#r_min = 2.4000
r_max = 4.2000
#dr    = 0.6000 #it is user's choice
#r_thr =15.0000 #threshhold r value, also choice

with open('Tet_OP.txt', 'w') as f:
    t = 0.0
    for ts in u.trajectory:
        t = t + 1
        solv_shell_4A = u.atoms.select_atoms('name OW and around 4.2 protein')

        for hyd_water in solv_shell_4A:
            ne_water = u.select_atoms(f'name OW and around {r_max} index {hyd_water.id}')
            if(len(ne_water)) < 4: """If the number of waters within 4.2A is less than 4 (coordination number), skip the particular water molecule/step"""
                continue
            else:
                R_n = ne_water.positions
                R_0 = np.zeros(R_n.shape,dtype=np.float32)
                for i in range(len(R_n)):
                    for j in range(3):
                        R_0[i][j] = hyd_water.position[j]
                R_rel = R_n - R_0
                dist_R_rel = dist_array(R_rel)
                dist_nearest_4 = dist_R_rel[dist_R_rel[:,0].argsort()][0:4]
                R_4 = dist_nearest_4[:,1:]
                d_4 = dist_nearest_4[:,0:1]
                """orientational tetrahedral order parameter"""
                cos_sum = 0.0
                for i in range(3):
                    for j in range(i+1,4,1):
                        cos = cos_sim(R_4[i],R_4[j])
                        cos_sum = cos_sum + (cos + 1.0/3.0)*(cos + 1.0/3.0)
                q = 1.0 - (3.0/8.0)*cos_sum


                f.write(f'{t}\t{S_k(d_4)}\t{q}')
                f.write('\n')
